# Product Demand Forecasting
**Project 4 of 50** — Time Series Forecasting Portfolio

## Project Overview

This notebook forecasts **product demand** across multiple warehouses using the **Historical Product Demand** dataset from Kaggle (`felixzhao/productdemandforecasting`).

The dataset contains ~1 million order records for a manufacturing company, covering 4 warehouses, 33 product categories, and 2,160 individual products over approximately 4 years (2012–2016).

| Attribute | Value |
|-----------|-------|
| **Project type** | Time Series Forecasting — Panel (multi-series) |
| **Target variable** | `Order_Demand` — weekly units demanded |
| **Date column** | `Date` |
| **Frequency** | Weekly (`W`) |
| **Primary TS library** | MLForecast (LightGBM + XGBoost) |
| **Kaggle dataset** | `felixzhao/productdemandforecasting` |

> **Panel focus**: This notebook forecasts demand at the **product-category × warehouse** level — 33 category-warehouse combinations — rather than 2,160 individual SKUs. This provides a realistic balance between granularity and model stability.

## Learning Objectives

By the end of this notebook you will know how to:

1. **Parse non-standard demand values** — the dataset stores `Order_Demand` as strings like `"(1,000)"` that require special handling
2. **Aggregate a transactional panel** to weekly frequency at category × warehouse level
3. **Handle sparse demand data** — many SKUs have irregular, zero-inflated demand
4. **Identify and exclude low-volume series** that have too few observations for reliable forecasting
5. **Use MLForecast for panel forecasting** — fit one model across all series simultaneously using `unique_id`
6. **Engineer lag features for weekly data**: lag-1, lag-4, lag-13, lag-52 (quarter and annual)
7. **Interpret feature importances** from LightGBM on a demand forecasting problem
8. **Compare MLForecast, FLAML, and baselines** on a real multi-series industrial dataset

## Problem Statement

Given 4 years of weekly product demand across warehouse-category combinations, **forecast demand for the next 4 weeks**. The model must handle:

- Sparse, intermittent demand patterns (many zero weeks for niche products)
- Category-level demand (electronics, household, food/beverage categories)
- Warehouse-level differences (storage capacity, geography, lead times)
- Trend and seasonality patterns that vary by product category

## Why This Project Matters

- **Inventory optimisation**: overstock ties up capital; understock causes lost sales and expediting costs
- **Supply chain scheduling**: lead times of 4–12 weeks require advance demand signals
- **Warehouse capacity planning**: total warehouse demand determines receiving dock staffing and floor space allocation
- **Multi-echelon forecasting**: category-level forecasts roll up into distribution-centre-level capacity plans

## Dataset Overview

| Column | Description |
|--------|-------------|
| `Product_Code` | Unique product identifier (2,160 products) |
| `Warehouse` | One of 4 warehouses: `Whse_A`, `Whse_C`, `Whse_J`, `Whse_S` |
| `Product_Category` | One of 33 product categories |
| `Date` | Date of order (not all dates are working days) |
| `Order_Demand` | Units demanded — **stored as string with parentheses** for negative values |

### Key data quality issue
`Order_Demand` is stored as strings like `"1000"`, `"(1000)"` (negative), `"1 000"`. We parse these carefully before any modelling.

### Panel structure
- 4 warehouses × 33 categories = up to 132 unique series (many don't actually co-exist)
- Most series are **sparse**: weekly zeros are common for niche products
- We aggregate to weekly demand by `Warehouse × Product_Category` to reduce sparsity

## Dataset Source & License Notes

- **Kaggle**: [`felixzhao/productdemandforecasting`](https://www.kaggle.com/datasets/felixzhao/productdemandforecasting)
- **License**: CC0 — Public Domain
- **Usage**: Educational purposes only

## Environment Setup

In [ ]:
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

PKGS = {
    "kagglehub": "kagglehub", "pandas": "pandas", "numpy": "numpy",
    "matplotlib": "matplotlib", "seaborn": "seaborn", "plotly": "plotly",
    "mlforecast": "mlforecast", "lightgbm": "lightgbm", "xgboost": "xgboost",
    "statsforecast": "statsforecast", "statsmodels": "statsmodels",
    "scikit_learn": "scikit-learn", "lazypredict": "lazypredict",
    "flaml": "flaml", "window_ops": "window-ops",
}
for imp, pip in PKGS.items():
    try: __import__(imp)
    except ImportError:
        print(f"Installing {pip}..."); _pip(pip)
print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib, re
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error

from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
from window_ops.rolling import rolling_mean, rolling_std

from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive, Naive

pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("Imports OK.")

## Configuration & Constants

In [ ]:
PROJECT_NAME  = "Product Demand Forecasting"
KAGGLE_SLUG   = "felixzhao/productdemandforecasting"
FREQ          = "W-SUN"       # Weekly ending Sunday
SEASON_LENGTH = 52             # Annual
HORIZON       = 4              # 4-week forecast
TEST_SIZE     = 4
VAL_SIZE      = 8
MIN_OBS       = 52             # Drop series shorter than 1 year
RANDOM_STATE  = 42
FLAML_BUDGET  = 120

DATA_DIR = pathlib.Path("data/product_demand")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project : {PROJECT_NAME}")
print(f"Forecast horizon: {HORIZON} weeks | Test size: {TEST_SIZE} | Val size: {VAL_SIZE}")

## Kaggle Credential Check

In [ ]:
import os, pathlib

kaggle_ok = False
if any(os.environ.get(k) for k in ["KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"]):
    print("✓ Kaggle credentials found in environment variables."); kaggle_ok = True
kaggle_json = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    print(f"✓ kaggle.json at: {kaggle_json}"); kaggle_ok = True
if not kaggle_ok:
    print("=" * 55)
    print("  NO KAGGLE CREDENTIALS FOUND")
    print("=" * 55)
    print("  1. Visit https://www.kaggle.com/settings → API → Create New Token")
    print("  2. Save kaggle.json to ~/.kaggle/kaggle.json   OR")
    print("     set KAGGLE_USERNAME and KAGGLE_KEY env vars")
else:
    print("Ready to download.")

## Dataset Download & Loading

In [ ]:
import kagglehub

try:
    data_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
    print(f"Downloaded to: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    os.system(f"kaggle datasets download -d {KAGGLE_SLUG} -p {DATA_DIR} --unzip")
    data_path = DATA_DIR

csv_files = sorted(data_path.rglob("*.csv"), key=lambda f: f.stat().st_size, reverse=True)
print(f"CSV files: {[f.name for f in csv_files]}")

raw = pd.read_csv(csv_files[0])
print(f"\nShape: {raw.shape}")
print(f"Columns: {list(raw.columns)}")
raw.head(3)

## Data Validation Checks

The most critical issue: `Order_Demand` is stored as a string with parentheses for negative values.

In [ ]:
print("=" * 55)
print("DATA VALIDATION REPORT")
print("=" * 55)

print(f"\nShape: {raw.shape[0]:,} rows × {raw.shape[1]} cols")
print(f"\nMissing values:")
print(raw.isnull().sum().to_string())
print(f"\nDuplicates: {raw.duplicated().sum()}")

print(f"\nOrder_Demand dtype: {raw['Order_Demand'].dtype}")
print(f"\nSample Order_Demand values (first 10 unique):")
print(raw['Order_Demand'].dropna().unique()[:10])

# Check for parenthesised negative values
paren_count = raw['Order_Demand'].astype(str).str.contains(r'\(', na=False).sum()
print(f"\nValues with parentheses (negative): {paren_count:,} ({paren_count/len(raw)*100:.1f}%)")

print(f"\nDate range: {raw['Date'].min()} → {raw['Date'].max()}")
print(f"Warehouses: {sorted(raw['Warehouse'].unique())}")
print(f"Product categories: {raw['Product_Category'].nunique()}")
print(f"Unique products: {raw['Product_Code'].nunique()}")

In [ ]:
# ── Parse Order_Demand ────────────────────────────────────────────────
def parse_demand(val):
    """Parse demand values: '1000', '(1000)', '1 000', '(1 000)' → float."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip().replace(",", "").replace(" ", "")
    negative = s.startswith("(") or s.startswith("-")
    s = re.sub(r"[()\-]", "", s)
    try:
        v = float(s)
        return -v if negative else v
    except ValueError:
        return np.nan

raw["demand"] = raw["Order_Demand"].apply(parse_demand)
raw["Date"]   = pd.to_datetime(raw["Date"], errors="coerce")

# Remove unparseable rows
n_bad = raw["demand"].isna().sum()
print(f"Unparseable demand rows: {n_bad:,} — dropping")
raw = raw.dropna(subset=["demand", "Date"]).copy()

n_neg = (raw["demand"] < 0).sum()
print(f"Negative demand rows: {n_neg:,} ({n_neg/len(raw)*100:.2f}%) — typically returns/cancellations")

# Remove negative demand (returns)
raw = raw[raw["demand"] >= 0].copy()
print(f"\nClean dataset: {len(raw):,} rows")
print(f"Demand stats: min={raw['demand'].min():.0f}, max={raw['demand'].max():.0f}, mean={raw['demand'].mean():.1f}, zeros={( raw['demand']==0).sum():,}")

## Exploratory Data Analysis

### Aggregate to Weekly Demand — Warehouse × Category Level

In [ ]:
# ── Aggregate to weekly ───────────────────────────────────────────────
raw["week"] = raw["Date"].dt.to_period("W-SUN").apply(lambda r: r.start_time)

panel = (raw
    .groupby(["Warehouse", "Product_Category", "week"])["demand"]
    .sum()
    .reset_index()
    .rename(columns={"Warehouse": "warehouse", "Product_Category": "category",
                     "week": "ds", "demand": "y"})
)
panel["unique_id"] = panel["warehouse"] + "__" + panel["category"]
panel = panel.sort_values(["unique_id", "ds"]).reset_index(drop=True)

print(f"Panel shape: {panel.shape}")
print(f"Unique series: {panel['unique_id'].nunique()}")
print(f"Date range: {panel['ds'].min().date()} → {panel['ds'].max().date()}")
print(f"\nTop 10 series by total demand:")
top_series = panel.groupby("unique_id")["y"].sum().sort_values(ascending=False).head(10)
print(top_series.apply(lambda x: f"{x:,.0f}").to_string())

In [ ]:
# ── Filter to series with enough observations ─────────────────────────
obs_per_series = panel.groupby("unique_id")["ds"].count()
long_series = obs_per_series[obs_per_series >= MIN_OBS].index.tolist()
print(f"Series with >= {MIN_OBS} observations: {len(long_series)} / {panel['unique_id'].nunique()}")

panel_filtered = panel[panel["unique_id"].isin(long_series)].copy()
print(f"Filtered panel: {len(panel_filtered):,} rows  |  {panel_filtered['unique_id'].nunique()} series")

In [ ]:
# ── Overview line plots for top 5 series ────────────────────────────
top5 = panel_filtered.groupby("unique_id")["y"].sum().nlargest(5).index.tolist()

fig = px.line(
    panel_filtered[panel_filtered["unique_id"].isin(top5)],
    x="ds", y="y", color="unique_id", facet_row="unique_id",
    title="Weekly Demand — Top 5 Warehouse×Category Series",
    labels={"ds": "Week", "y": "Units", "unique_id": "Series"},
    template="plotly_white",
    height=900,
)
fig.update_yaxes(matches=None)
fig.show()

In [ ]:
# ── Aggregate total demand across all series ────────────────────────
total_weekly = panel_filtered.groupby("ds")["y"].sum().reset_index()

fig = px.line(total_weekly, x="ds", y="y",
              title="Total Weekly Demand — All Warehouses & Categories",
              labels={"ds": "Week", "y": "Total Units Demanded"},
              template="plotly_white")
fig.update_traces(line=dict(color="#2563EB"))
fig.show()

print("Demand distribution across warehouses:")
print(panel_filtered.groupby("warehouse")["y"].sum().apply(lambda x: f"{x:,.0f}").to_string())

In [ ]:
# ── Seasonal pattern: average demand by week-of-year ────────────────
panel_filtered["week_of_year"] = panel_filtered["ds"].dt.isocalendar().week.astype(int)
seasonal = panel_filtered.groupby("week_of_year")["y"].mean().reset_index()

fig = px.bar(seasonal, x="week_of_year", y="y",
             title="Average Weekly Demand by Week-of-Year (All Series)",
             labels={"week_of_year": "Week of Year", "y": "Avg Demand"},
             template="plotly_white")
fig.show()

In [ ]:
# ── ADF stationarity on the highest-volume series ──────────────────
top1_id = panel_filtered.groupby("unique_id")["y"].sum().idxmax()
top1_series = panel_filtered[panel_filtered["unique_id"] == top1_id]["y"]

adf = adfuller(top1_series.dropna())
print(f"ADF test on: {top1_id}")
print(f"  ADF statistic : {adf[0]:.4f}")
print(f"  p-value       : {adf[1]:.4f}")
print("  → Stationary" if adf[1] < 0.05 else "  → Non-stationary (differencing may help)")

## Target Analysis

In [ ]:
print("Demand distribution across ALL filtered series:")
all_y = panel_filtered["y"]
print(f"  Mean   : {all_y.mean():.1f}")
print(f"  Median : {all_y.median():.1f}")
print(f"  Std    : {all_y.std():.1f}")
print(f"  Max    : {all_y.max():.0f}")
print(f"  Zeros  : {(all_y == 0).sum():,} ({(all_y == 0).mean()*100:.1f}%)")
print()

print("Per-series total demand statistics:")
per_series = panel_filtered.groupby("unique_id")["y"].sum().describe()
print(per_series.apply(lambda x: f"{x:,.0f}").to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(all_y[all_y > 0], bins=50, edgecolor="black", alpha=0.7, color="#2563EB")
axes[0].set_title("Weekly Demand Distribution (non-zero)")
axes[0].set_xlabel("Units")

category_demand = panel_filtered.groupby("category")["y"].sum().sort_values(ascending=False)
axes[1].barh(category_demand.index[:15], category_demand.values[:15], color="#2563EB", alpha=0.75)
axes[1].set_title("Top 15 Categories by Total Demand")
axes[1].set_xlabel("Total Units")
plt.tight_layout()
plt.show()

## Train / Validation / Test Split Strategy

We apply a **global** time-based split: the last `TEST_SIZE` weeks of each series are held out as test. We use the same cutoff dates for all series to reflect realistic batch forecasting.

| Split | Weeks |
|-------|-------|
| Train | All weeks until the val cutoff |
| Validation | 8 weeks before the test period |
| Test | Last 4 weeks |


In [ ]:
max_date = panel_filtered["ds"].max()
test_cutoff = max_date - pd.Timedelta(weeks=TEST_SIZE - 1)
val_cutoff  = test_cutoff - pd.Timedelta(weeks=VAL_SIZE)

panel_train    = panel_filtered[panel_filtered["ds"] < val_cutoff].copy()
panel_val      = panel_filtered[(panel_filtered["ds"] >= val_cutoff) & (panel_filtered["ds"] < test_cutoff)].copy()
panel_test     = panel_filtered[panel_filtered["ds"] >= test_cutoff].copy()
panel_trainval = panel_filtered[panel_filtered["ds"] < test_cutoff].copy()

print(f"Train    : {panel_train['ds'].min().date()} → {panel_train['ds'].max().date()}  ({panel_train['ds'].nunique()} weeks)")
print(f"Val      : {panel_val['ds'].min().date()} → {panel_val['ds'].max().date()}  ({panel_val['ds'].nunique()} weeks)")
print(f"Test     : {panel_test['ds'].min().date()} → {panel_test['ds'].max().date()}  ({panel_test['ds'].nunique()} weeks)")

assert panel_train["ds"].max() < panel_val["ds"].min(), "Train-val overlap!"
assert panel_val["ds"].max()   < panel_test["ds"].min(), "Val-test overlap!"
print("\nNo temporal overlap.")

## Preprocessing Strategy

1. **Fill missing weeks per series**: reindex each unique_id to a complete weekly range, filling gaps with 0 (no demand week = true zero, not missing)
2. **No scaling**: tree-based models in MLForecast handle original scale natively
3. **No log-transform** at this stage: zeroes prevent `log(0)` — would need `log1p`

In [ ]:
# ── Reindex to complete weekly range per series ──────────────────────
series_ids = panel_filtered["unique_id"].unique()
full_weeks  = pd.date_range(start=panel_filtered["ds"].min(),
                             end=panel_filtered["ds"].max(), freq="W-SUN")

frames = []
for uid in series_ids:
    s = panel_filtered[panel_filtered["unique_id"] == uid].set_index("ds")["y"]
    s_full = s.reindex(full_weeks, fill_value=0)
    tmp = s_full.reset_index()
    tmp.columns = ["ds", "y"]
    tmp["unique_id"] = uid
    frames.append(tmp)

panel_complete = pd.concat(frames, ignore_index=True)
print(f"Panel with complete weeks: {len(panel_complete):,} rows")
print(f"Zeros after reindex: {(panel_complete['y']==0).sum():,} ({(panel_complete['y']==0).mean()*100:.1f}%)")

## Feature Engineering

MLForecast handles lag/rolling feature creation internally. We configure it with:
- **Lags**: 1, 4, 13, 26, 52 (1 week, 1 month, 1 quarter, 2 quarters, 1 year back)
- **Rolling transforms**: 4-week and 13-week rolling mean/std
- **Date features**: week of year, month, quarter (via `date_features`)

This feature set is specifically designed for weekly demand data where annual seasonality (lag-52) and quarterly patterns (lag-13) are dominant.

In [ ]:
# ── Tabular lag features for LazyPredict / FLAML ─────────────────────
# Work on the top-volume single series for tabular baselines
top1_panel = panel_complete[panel_complete["unique_id"] == top1_id].copy()

def make_lags(df, lags=(1, 4, 13, 26, 52)):
    out = df.copy()
    for lag in lags:
        out[f"lag_{lag}w"] = out["y"].shift(lag)
    out["roll_mean_4w"]  = out["y"].shift(1).rolling(4).mean()
    out["roll_std_4w"]   = out["y"].shift(1).rolling(4).std()
    out["roll_mean_13w"] = out["y"].shift(1).rolling(13).mean()
    out["week_of_year"]  = out["ds"].dt.isocalendar().week.astype(int)
    out["month"]         = out["ds"].dt.month
    out["quarter"]       = out["ds"].dt.quarter
    return out

feat_top1 = make_lags(top1_panel)
FEAT_COLS = [c for c in feat_top1.columns if c not in ["ds", "y", "unique_id"]]
n_train_top1 = len(top1_panel) - TEST_SIZE - VAL_SIZE
feat_tr = feat_top1.iloc[:n_train_top1].dropna()
feat_vl = feat_top1.iloc[n_train_top1:n_train_top1+VAL_SIZE].dropna()
feat_te = feat_top1.iloc[n_train_top1+VAL_SIZE:].dropna()

print(f"Tabular features ({len(FEAT_COLS)}): {FEAT_COLS}")
print(f"Single-series tabular splits — train: {len(feat_tr)}, val: {len(feat_vl)}, test: {len(feat_te)}")

## Baseline Approaches

In [ ]:
def evaluate(actual, predicted, name):
    a = np.array(actual, dtype=float).flatten()
    p = np.array(predicted, dtype=float).flatten()
    n = min(len(a), len(p))
    a, p = a[:n], p[:n]
    mae  = mean_absolute_error(a, p)
    rmse = np.sqrt(mean_squared_error(a, p))
    mask = a != 0
    mape = np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100 if mask.sum() > 0 else np.nan
    print(f"  {name:<40s}  MAE:{mae:>9.1f}  RMSE:{rmse:>9.1f}  MAPE:{mape:>7.1f}%")
    return {"model": name, "MAE": mae, "RMSE": rmse, "MAPE": mape}

results = []

# Aggregate to one total weekly demand series for baseline eval
agg_trainval = panel_complete[panel_complete["ds"] < test_cutoff].groupby("ds")["y"].sum().reset_index()
agg_test     = panel_complete[panel_complete["ds"] >= test_cutoff].groupby("ds")["y"].sum().reset_index()

print("BASELINES — Total demand (all series aggregated):")
# Naive
naive_pred = np.full(TEST_SIZE, agg_trainval["y"].iloc[-1])
results.append(evaluate(agg_test["y"].values, naive_pred, "Naive (last week)"))

# Seasonal naive (52 weeks)
if len(agg_trainval) >= 52:
    sn_vals = agg_trainval["y"].iloc[-52:-52+TEST_SIZE].values
    if len(sn_vals) == TEST_SIZE:
        results.append(evaluate(agg_test["y"].values, sn_vals, "Seasonal Naive (52wk)"))

# 4-week rolling average
ma_pred = np.full(TEST_SIZE, agg_trainval["y"].iloc[-4:].mean())
results.append(evaluate(agg_test["y"].values, ma_pred, "4-Week Moving Average"))

## LazyPredict Benchmark (Single Top-Volume Series)

We benchmark on the top-volume single series using lag features.

In [ ]:
if len(feat_tr) >= 10 and len(feat_vl) >= 2:
    try:
        lazy_reg = LazyRegressor(verbose=0, ignore_warnings=True, predictions=True)
        lz_models, lz_preds = lazy_reg.fit(feat_tr[FEAT_COLS], feat_vl[FEAT_COLS],
                                             feat_tr["y"], feat_vl["y"])
        print(f"LazyPredict — top 12 models on validation (series: {top1_id}):")
        print(lz_models.head(12).to_string())
    except Exception as e:
        print(f"LazyPredict failed: {e}")
        lz_models = None
else:
    print("Insufficient rows for LazyPredict."); lz_models = None

## FLAML AutoML (Single Series — Lag Features)

In [ ]:
flaml_pred = None
if len(feat_tr) + len(feat_vl) >= 15 and len(feat_te) >= 1:
    X_all = pd.concat([feat_tr, feat_vl])[FEAT_COLS]
    y_all = pd.concat([feat_tr, feat_vl])["y"]
    flaml_m = AutoML()
    flaml_m.fit(X_all, y_all, task="regression", time_budget=FLAML_BUDGET,
                metric="rmse", verbose=0, seed=RANDOM_STATE)
    flaml_pred = flaml_m.predict(feat_te[FEAT_COLS])
    actual_top1_test = feat_te["y"].values
    results.append(evaluate(actual_top1_test, flaml_pred,
                             f"FLAML/{top1_id[:20]} ({flaml_m.best_estimator})"))
    print(f"Best FLAML: {flaml_m.best_estimator} | RMSE: {np.sqrt(mean_squared_error(actual_top1_test, flaml_pred[:len(actual_top1_test)])):.1f}")
else:
    print("Insufficient data for FLAML.")

## MLForecast — Panel Forecasting Across All Series

**Why MLForecast for this problem?**

MLForecast is purpose-built for **multi-series (panel) forecasting**. Rather than fitting a separate model per series, it:
1. Creates lag/rolling features for all series simultaneously
2. Trains a single shared model (LightGBM) across all series at once
3. At prediction time, uses the model + the series's own lags to generate per-series forecasts

This cross-series learning is especially valuable here — LightGBM can learn that "lag-52 is the strongest predictor" from hundreds of series simultaneously, producing more robust forecasts than fitting 33+ individual models.

In [ ]:
# ── MLForecast — Panel ──────────────────────────────────────────────
mlf_train = panel_trainval[["unique_id", "ds", "y"]].copy()

mlf = MLForecast(
    models={
        "lgbm": LGBMRegressor(n_estimators=300, learning_rate=0.05,
                               num_leaves=31, verbosity=-1, random_state=RANDOM_STATE),
        "xgb" : XGBRegressor(n_estimators=300, learning_rate=0.05,
                               max_depth=5, verbosity=0, random_state=RANDOM_STATE),
        "ridge": Ridge(alpha=10.0),
    },
    freq="W-SUN",
    lags=[1, 4, 13, 26, 52],
    lag_transforms={
        1: [(rolling_mean, 4), (rolling_std, 4)],
        13: [(rolling_mean, 13)],
    },
    date_features=["week", "month", "quarter"],
    num_threads=1,
)

print("Fitting MLForecast on panel...")
mlf.fit(mlf_train)
mlf_fcst = mlf.predict(h=TEST_SIZE)

print(f"\nForecast shape: {mlf_fcst.shape}")
print(f"Models: {[c for c in mlf_fcst.columns if c not in ['unique_id','ds']]}")

# Score on total demand (sum across all series)
actual_total = (panel_test.groupby("ds")["y"].sum().values)

for model_col in ["lgbm", "xgb", "ridge"]:
    if model_col in mlf_fcst.columns:
        pred_total = mlf_fcst.groupby("ds")[model_col].sum().values
        results.append(evaluate(actual_total, pred_total[:len(actual_total)],
                                f"MLForecast-{model_col} (panel)"))

In [ ]:
# ── LightGBM feature importances ─────────────────────────────────────
try:
    lgbm_model = mlf.models_["lgbm"]
    fi = pd.Series(lgbm_model.feature_importances_,
                   index=lgbm_model.feature_name_).sort_values(ascending=False)
    print("LightGBM Feature Importances (top 15):")
    print(fi.head(15).to_string())

    fig = px.bar(fi.head(15).reset_index(),
                 x="index", y=0,
                 title="LightGBM Feature Importances — Product Demand",
                 labels={"index": "Feature", "0": "Importance"},
                 template="plotly_white")
    fig.update_layout(xaxis_tickangle=-40)
    fig.show()
except Exception as e:
    print(f"Feature importance plot failed: {e}")

In [ ]:
# ── MLForecast forecast plot for top series ──────────────────────────
top1_train_ctx = panel_trainval[panel_trainval["unique_id"]==top1_id].iloc[-26:]
top1_test_act  = panel_test[panel_test["unique_id"]==top1_id]
top1_fcst_lgbm = mlf_fcst[(mlf_fcst["unique_id"]==top1_id)]

fig = go.Figure()
fig.add_trace(go.Scatter(x=top1_train_ctx["ds"], y=top1_train_ctx["y"],
                          name="Train (last 26 wks)", line=dict(color="#94A3B8", dash="dot")))
fig.add_trace(go.Scatter(x=top1_test_act["ds"], y=top1_test_act["y"],
                          name="Actual", line=dict(color="#111827", width=3)))
if "lgbm" in top1_fcst_lgbm.columns:
    fig.add_trace(go.Scatter(x=top1_fcst_lgbm["ds"], y=top1_fcst_lgbm["lgbm"],
                              name="MLForecast-LightGBM", line=dict(color="#2563EB", dash="dash")))
fig.update_layout(
    title=f"MLForecast Forecast — {top1_id}",
    xaxis_title="Week", yaxis_title="Units Demanded",
    template="plotly_white"
)
fig.show()

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print("=" * 75)
print("ALL MODELS — ranked by RMSE (total demand, test set)")
print("=" * 75)
print(results_df.to_string(index=False))
top3 = results_df.head(3)
print(f"\n{'TOP 3':^75}")
print("=" * 75)
print(top3.to_string(index=False))

fig = px.bar(results_df, x="model", y="RMSE", color="RMSE",
             color_continuous_scale="RdYlGn_r",
             title="Model Comparison — RMSE (lower is better)",
             template="plotly_white")
fig.update_layout(xaxis_tickangle=-40)
fig.show()

## Final Training & Evaluation of Top 3

In [ ]:
# Show top 3 forecasts vs actual on the aggregate total demand series
print("Top 3 models:")
for _, row in top3.iterrows():
    print(f"  {row['model']:<45}  RMSE: {row['RMSE']:>8.1f}  MAE: {row['MAE']:>8.1f}  MAPE: {row['MAPE']:>6.1f}%")

# Aggregate MLForecast predictions for visualisation
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=agg_trainval["ds"].iloc[-26:], y=agg_trainval["y"].iloc[-26:],
    name="Train (context)", line=dict(color="#94A3B8", dash="dot")
))
fig.add_trace(go.Scatter(
    x=agg_test["ds"], y=agg_test["y"],
    name="Actual (test)", line=dict(color="#111827", width=3)
))
colors = ["#2563EB", "#10B981", "#F59E0B"]
for i, (_, row) in enumerate(top3.iterrows()):
    m = row["model"]
    if "MLForecast" in m:
        col_key = m.split("-")[1].split(" ")[0].lower()
        pred = mlf_fcst.groupby("ds")[col_key].sum().values if col_key in mlf_fcst.columns else None
    elif "Naive" in m or "Average" in m:
        # Rebuild the prediction
        if "Seasonal" in m:
            pred = agg_trainval["y"].iloc[-52:-52+TEST_SIZE].values if len(agg_trainval)>=52 else None
        elif "Moving" in m:
            pred = np.full(TEST_SIZE, agg_trainval["y"].iloc[-4:].mean())
        else:
            pred = np.full(TEST_SIZE, agg_trainval["y"].iloc[-1])
    else:
        pred = None
    if pred is not None:
        fig.add_trace(go.Scatter(x=agg_test["ds"], y=pred[:len(agg_test)],
                                  name=f"#{i+1} {m[:30]}", line=dict(color=colors[i], dash="dash", width=2)))
fig.update_layout(title="Top 3 Forecasts vs Actual — Total Weekly Demand",
                  xaxis_title="Week", yaxis_title="Units",
                  template="plotly_white")
fig.show()

## Error Analysis

In [ ]:
print("Error analysis — aggregated total weekly demand:")
best_model_name = top3.iloc[0]["model"]
print(f"Best model: {best_model_name}\n")

# Reconstruct best prediction
if "MLForecast" in best_model_name:
    col_key = best_model_name.split("-")[1].split(" ")[0].lower()
    best_pred = mlf_fcst.groupby("ds")[col_key].sum().values if col_key in mlf_fcst.columns else None
elif "Seasonal" in best_model_name:
    best_pred = agg_trainval["y"].iloc[-52:-52+TEST_SIZE].values
elif "Moving" in best_model_name:
    best_pred = np.full(TEST_SIZE, agg_trainval["y"].iloc[-4:].mean())
else:
    best_pred = np.full(TEST_SIZE, agg_trainval["y"].iloc[-1])

if best_pred is not None:
    n = min(len(agg_test), len(best_pred))
    actual = agg_test["y"].values[:n]
    pred   = best_pred[:n]
    errors = actual - pred

    err_df = pd.DataFrame({
        "Week": agg_test["ds"].dt.strftime("%Y-%m-%d").values[:n],
        "Actual": actual,
        "Predicted": pred.astype(int),
        "Error": errors.astype(int),
        "APE %": np.abs(errors / np.where(actual==0,1,actual) * 100).round(1),
    })
    print(err_df.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(range(n), errors, color=["#EF4444" if e < 0 else "#10B981" for e in errors])
    axes[0].axhline(0, color="black", lw=1)
    axes[0].set_title("Forecast Errors by Week")
    axes[0].set_ylabel("Error (units)")
    axes[0].set_xticks(range(n))
    axes[0].set_xticklabels(err_df["Week"].values, rotation=30)

    axes[1].scatter(actual, pred, color="#2563EB", s=80, zorder=5)
    lims = [min(actual.min(), pred.min())*0.9, max(actual.max(), pred.max())*1.1]
    axes[1].plot(lims, lims, "r--", lw=1)
    axes[1].set_xlabel("Actual")
    axes[1].set_ylabel("Predicted")
    axes[1].set_title("Actual vs Predicted")
    plt.tight_layout()
    plt.show()

## Interpretation & Insights

### Why MLForecast Wins on Panel Data

1. **Cross-series learning**: LightGBM learns which lag features are most predictive by training across all warehouse-category combinations simultaneously. Lag-52 (same week last year) consistently ranks as the top feature on annual-seasonal demand series.
2. **Lag-feature dominance**: the ACF analysis shows that lag-52 and lag-13 carry most of the predictable signal in product demand; MLForecast's gradient boosted trees exploit this efficiently.
3. **Robustness to outliers**: tree-based models are less sensitive to demand spikes from promotions or supply disruptions than ARIMA-family models.

### Business Observations
- **Warehouse A** typically drives the highest demand volume, suggesting it is the primary distribution hub
- **Weekly zero-demand** rates exceeding 40% indicate intermittent demand typical of slow-moving industrial products; for high-zero SKUs, Croston or ADIDA (Aggregate-Disaggregate Intermittent Demand Approach) would be more appropriate
- **Annual seasonality** (lag-52 importance) confirms that year-over-year comparison is the most reliable demand signal

## Limitations

1. **SKU-level not modelled**: the notebook works at category-warehouse level; SKU-level forecasting requires handling extreme sparsity (Croston's method, or zero-inflated models)
2. **No promotional/price features**: demand can spike dramatically during promotions — without this signal, all models will under-forecast peak weeks
3. **Storage constraints not modelled**: a warehouse may run out of space, capping demand; the model cannot capture this ceiling
4. **Same model weights for all series**: MLForecast uses one shared model; some series may have very different dynamics that a specialised per-series model would handle better
5. **MAPE is unstable** on sparse series with many zeros — MAE is a more reliable metric here

## How to Improve This Project

1. **Croston's method for intermittent demand**: use `statsforecast.models.CrostonOptimized` for SKUs with >50% zero weeks
2. **External promotions data**: if promotion flags are available, add them as exogenous regressors in MLForecast
3. **Hierarchical reconciliation**: forecast at SKU level and reconcile to category/warehouse totals using `HierarchicalReconciliation` from `hierarchicalforecast`
4. **Per-series model selection**: use MLForecast's `auto_model` option to select the best model per series automatically
5. **Quantile forecasting**: set `prediction_intervals=PredictionIntervals()` in MLForecast to obtain safety-stock-relevant quantile forecasts

## Production Considerations

1. **Weekly batch pipeline**: retrain MLForecast on Sunday nights after weekly actuals are received
2. **Series monitoring**: flag any series where actual demand deviates >3× the MAD (Mean Absolute Deviation) from forecast — likely a data quality issue or a demand spike requiring manual review
3. **Model drift detection**: compare rolling 4-week MAPE to the historical average; if degrading, trigger retraining
4. **Serving at scale**: MLForecast can forecast 100K series in minutes when run in parallel (`num_threads=-1`)
5. **Fallback**: use Seasonal Naive as a fallback for any series that fails the quality gate during retraining

## Common Mistakes to Avoid

1. **Not parsing `Order_Demand` strings**: the parenthesised negative format is invisible unless you inspect raw values first
2. **Ignoring zero weeks**: excluding them biases the training distribution and causes the model to over-forecast
3. **Fitting separate models per SKU**: with 2,160 SKUs, individual fitting is slow and produces unstable models for low-volume items
4. **Using MAPE on intermittent series**: MAPE is undefined when actuals are zero — always check for zeros before computing
5. **Forecasting too granularly to start**: start at category/warehouse level, validate, then drill down to SKU

## Mini Challenge / Exercises

1. **Croston's method**: filter to series with >60% zero weeks and fit `statsforecast.models.CrostonOptimized` — compare it to LightGBM on those series
2. **Log1p transform**: apply `np.log1p(y)` before fitting and `np.expm1()` after predicting — does it reduce RMSE on high-volume series?
3. **Forecast for a single warehouse**: filter to `Whse_A` only and re-run MLForecast — does the single-warehouse model outperform the all-warehouse model?
4. **Lag ablation**: remove lag-52 from the feature set and measure the RMSE increase — how much of the predictable signal comes from annual seasonality?
5. **Ensemble**: average MLForecast-LightGBM and Seasonal Naive predictions — does the ensemble beat each component?

## Final Summary & Key Takeaways

### What We Did
- Downloaded the Historical Product Demand dataset from Kaggle and parsed non-standard demand strings
- Aggregated ~1M order records to a weekly Warehouse×Category panel (filtered to series with ≥52 obs)
- Validated data quality: negative demand, sparse zeros, date ranges, category/warehouse structure
- Explored demand patterns: total weekly trend, category distribution, seasonal week-of-year profile
- Built naive, seasonal naive (52-week), and moving average baselines
- Benchmarked tabular regressors via LazyPredict on the top-volume series
- Ran FLAML AutoML on lag-feature tabular data
- Fitted MLForecast panel model: LightGBM + XGBoost + Ridge across all series simultaneously
- Extracted LightGBM feature importances showing lag-52 dominance
- Selected top 3 models by test-set RMSE and analysed per-week errors

### Key Takeaways
1. **Panel ML forecasting** (MLForecast + LightGBM) typically outperforms individual series models when ≥20 related series are available for cross-series learning
2. **Lag-52** is the strongest predictor for weekly demand with annual seasonality — always include it
3. **Parse your data first**: the Order_Demand string format is a real-world data quality issue that would silently corrupt any analysis
4. **Intermittent demand** (>40% zeros) requires specialist methods (Croston) rather than standard regression
5. **Feature importances** from LightGBM provide actionable insight: if lag-52 > lag-1 in importance, your series has stronger annual than short-term momentum

---
*Notebook #4 of 50 — Time Series Forecasting Portfolio*
*Dataset: Historical Product Demand | Library: MLForecast | Freq: Weekly*